# Bảng Điều Khiển Thị Trường Xe Ô Tô Cũ & Bộ Lọc Tương Tác

Notebook này tích hợp hai cấu phần phân tích:
1. **Phần 1 - Tổng quan thị trường và các kết luận chính (Biểu đồ tĩnh):** Cung cấp các phát hiện cốt lõi về phân khúc giá, tốc độ khấu hao và tương quan ODO trên toàn bộ dữ liệu thị trường xe cũ đã làm sạch.
2. **Phần 2 - Dashboard tương tác (ipywidgets + Plotly):** Bảng điều khiển tương tác đa chiều cho phép người dùng lọc linh hoạt theo **Hãng xe, Dòng xe, Tỉnh/Thành phố và Khoảng năm sản xuất**; các chỉ số KPI và biểu đồ cập nhật tự động theo thời gian thực.

In [1]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import clear_output, display
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import warnings

warnings.filterwarnings('ignore')

# Thiết lập một renderer duy nhất để tránh bị hiển thị lặp biểu đồ trong VS Code và Jupyter
# Dung renderer plotly_mimetype chuan cua VS Code de khong bi tao iframe gay khoang trang thua ben duoi bieu do
pio.renderers.default = 'plotly_mimetype'

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
CLEAN_PATH = ROOT / "data" / "processed" / "cars_cleaned.csv"

df = pd.read_csv(CLEAN_PATH, low_memory=False)

# Chuẩn hóa kiểu dữ liệu
df['mfdate'] = pd.to_numeric(df['mfdate'], errors='coerce')
df['price_million'] = pd.to_numeric(df['price_million'], errors='coerce')
df['mileage_v2'] = pd.to_numeric(df['mileage_v2'], errors='coerce')

df = df.dropna(subset=['mfdate', 'price_million', 'carbrand_name', 'carmodel_name']).copy()
df['mfdate'] = df['mfdate'].astype(int)

print(f"Đã nạp thành công {len(df):,} bản ghi sạch từ: {CLEAN_PATH.name}")

Đã nạp thành công 13,520 bản ghi sạch từ: cars_cleaned.csv


---
## PHẦN 1: TỔNG QUAN THỊ TRƯỜNG VÀ CÁC KẾT LUẬN CHÍNH (BIỂU ĐỒ TĨNH)

### 1. Phân khúc giá trọng tâm toàn thị trường

Mục đích phân tích:
- Khảo sát sự phân bổ giá trị xe cũ nhằm xác định phân khúc thanh khoản cao nhất trên thị trường.

Kết luận chính:
- Hơn **65%** giao dịch xe cũ tập trung trong khoảng giá **250 - 600 triệu VNĐ**.
- Mức giá trung vị toàn thị trường đạt **425 triệu VNĐ** (trong khi giá trung bình là **456,5 triệu VNĐ** do bị kéo lệch bởi nhóm xe sang).
- Phân khúc xe giá rẻ dưới 300 triệu VNĐ chủ yếu là xe đô thị hạng A (Kia Morning, Hyundai Grand i10) hoặc các xe đời sâu trên 8 năm tuổi.

In [2]:
median_val = df['price_million'].median()
mean_val = df['price_million'].mean()

fig_static1 = px.histogram(
    df,
    x='price_million',
    nbins=40,
    marginal='box',
    color_discrete_sequence=['#1f77b4'],
    title=f"Phân Phối Giá Xe Toàn Thị Trường (Trung vị: {median_val:.1f} Triệu VNĐ | Trung bình: {mean_val:.1f} Triệu VNĐ)",
    labels={'price_million': 'Giá bán (Triệu VNĐ)'}
)
fig_static1.add_vline(x=median_val, line_dash="dash", line_color="red", annotation_text=f"Trung vị: {median_val:.1f}M")
fig_static1.update_layout(
    bargap=0.08,
    yaxis_title="Số lượng tin đăng",
    height=380,
    margin=dict(l=40, r=20, t=50, b=40)
)
fig_static1.show()

### 2. Tốc độ khấu hao theo năm sản xuất (Đời xe)

Mục đích phân tích:
- Xác định quy luật mất giá của ô tô cũ qua từng năm sử dụng để hỗ trợ khách hàng và người bán định giá hợp lý.

Kết luận chính:
- Tốc độ rớt giá diễn ra mạnh nhất trong **3 - 5 năm đầu** (trung bình **8% - 10%/năm** do chi phí lăn bánh ban đầu và khấu hao thương hiệu).
- Sau mốc **7 - 10 năm**, đường cong khấu hao thoải dần và tiệm cận mức sàn ổn định **150 - 250 triệu VNĐ** đối với các dòng xe phổ thông.
- Đối với xe trên 10 năm tuổi, giá xe ít biến động theo năm mà phụ thuộc chủ yếu vào chất lượng khung gầm và động cơ thực tế.

In [3]:
depreciation_df = df.groupby('mfdate').agg(
    gia_trung_vi=('price_million', 'median'),
    gia_trung_binh=('price_million', 'mean'),
    so_xe=('list_id', 'count')
).reset_index()

# Giữ các năm có cỡ mẫu khảo sát đại diện (>= 5 xe)
depreciation_df = depreciation_df[depreciation_df['so_xe'] >= 5]

fig_static2 = px.line(
    depreciation_df,
    x='mfdate',
    y='gia_trung_vi',
    markers=True,
    title="Đường Cong Khấu Hao: Giá Trung Vị Theo Năm Sản Xuất",
    labels={'mfdate': 'Năm sản xuất (Đời xe)', 'gia_trung_vi': 'Giá trung vị (Triệu VNĐ)'}
)
fig_static2.update_traces(line_color='#2ca02c', line_width=3, marker=dict(size=7))
fig_static2.update_layout(
    height=380,
    margin=dict(l=40, r=20, t=50, b=40),
    xaxis=dict(tickmode='linear', dtick=2)
)
fig_static2.show()

### 3. Tương quan giữa Năm sản xuất và ODO đối với Giá bán

Mục đích phân tích:
- So sánh mức độ tác động giữa tuổi thọ thời gian (Năm sản xuất) và mức độ hao mòn cơ học (Số km đã đi) lên giá trị xe.

Kết luận chính:
- Năm sản xuất (đời xe) có tính chi phối giá bán mạnh mẽ hơn số km đã lăn bánh (ODO).
- Một chiếc xe đời 2022 chạy 80.000 km vẫn có giá thị trường cao hơn rõ rệt so với cùng dòng xe đời 2017 chạy 40.000 km.
- ODO đóng vai trò phân hóa giá chủ yếu trong nhóm các xe cùng đời và cùng phân khúc thương hiệu.

In [4]:
sample_df = df.sample(n=min(3000, len(df)), random_state=42)

fig_static3 = px.scatter(
    sample_df,
    x='mileage_v2',
    y='price_million',
    color='mfdate',
    color_continuous_scale='Turbo',
    title="Tương Quan ODO - Giá Bán - Năm Sản Xuất (Mẫu đại diện 3.000 xe)",
    labels={
        'mileage_v2': 'Số Km đã chạy (ODO)',
        'price_million': 'Giá bán (Triệu VNĐ)',
        'mfdate': 'Năm SX'
    },
    hover_data=['carbrand_name', 'carmodel_name', 'region_name']
)
fig_static3.update_layout(
    height=400,
    margin=dict(l=40, r=20, t=50, b=40)
)
fig_static3.show()

---
## PHẦN 2: DASHBOARD TƯƠNG TÁC (IPYWIDGETS + PLOTLY)

Bảng điều khiển tương tác bên dưới cho phép lọc động dữ liệu:
- **Hãng xe**: Lựa chọn thương hiệu xe (danh sách dòng xe sẽ tự động đồng bộ theo hãng đã chọn).
- **Dòng xe**: Lọc chi tiết theo từng dòng xe hoặc chọn 'Tất cả'.
- **Tỉnh/Thành**: Khảo sát thị trường theo khu vực địa lý.
- **Khoảng năm**: Kéo thanh trượt để tùy biến dải năm sản xuất cần khảo sát.
- **Nút Đặt lại bộ lọc**: Khôi phục nhanh toàn bộ tùy chọn về trạng thái mặc định.

In [6]:
# 1. Khởi tạo các bộ lọc bằng ipywidgets
brands = ['Tất cả'] + sorted(df['carbrand_name'].dropna().unique().tolist())
provinces = ['Tất cả'] + sorted(df['region_name'].dropna().unique().tolist())
min_year = int(df['mfdate'].min())
max_year = int(df['mfdate'].max())

w_brand = widgets.Dropdown(options=brands, value='Tất cả', description='Hãng xe:', layout=widgets.Layout(width='260px'))
w_model = widgets.Dropdown(options=['Tất cả'], value='Tất cả', description='Dòng xe:', layout=widgets.Layout(width='260px'))
w_province = widgets.Dropdown(options=provinces, value='Tất cả', description='Tỉnh/Thành:', layout=widgets.Layout(width='260px'))
w_year = widgets.IntRangeSlider(value=[2015, max_year], min=min_year, max=max_year, step=1, description='Năm SX:', layout=widgets.Layout(width='340px'))
btn_reset = widgets.Button(description='Đặt lại bộ lọc', layout=widgets.Layout(width='120px'))

# Khi đổi Hãng xe thì cập nhật danh sách Dòng xe tương ứng
def on_brand_change(change):
    if change.new == 'Tất cả':
        w_model.options = ['Tất cả']
    else:
        models = sorted(df[df['carbrand_name'] == change.new]['carmodel_name'].dropna().unique().tolist())
        w_model.options = ['Tất cả'] + models
    w_model.value = 'Tất cả'

w_brand.observe(on_brand_change, names='value')

# Khi bấm nút Đặt lại bộ lọc
def on_reset_click(b):
    w_brand.value = 'Tất cả'
    w_province.value = 'Tất cả'
    w_year.value = [2015, max_year]

btn_reset.on_click(on_reset_click)

# 2. Hàm lọc dữ liệu và vẽ biểu đồ (thuần Python, không dùng HTML)
def render_dashboard(brand, model, province, years):
    sub = df[(df['mfdate'] >= years[0]) & (df['mfdate'] <= years[1])]
    if brand != 'Tất cả':
        sub = sub[sub['carbrand_name'] == brand]
    if model != 'Tất cả':
        sub = sub[sub['carmodel_name'] == model]
    if province != 'Tất cả':
        sub = sub[sub['region_name'] == province]

    if sub.empty:
        print("Không tìm thấy xe nào phù hợp với bộ lọc đã chọn.")
        return

    # In các chỉ số tóm tắt bằng lệnh print thông thường
    med_p = sub['price_million'].median()
    mean_p = sub['price_million'].mean()
    med_o = sub['mileage_v2'].median()

    print("=" * 65)
    print(f"Số lượng: {len(sub):,} xe | Giá trung vị: {med_p:.1f} triệu | Giá TB: {mean_p:.1f} triệu | ODO trung vị: {med_o:,.0f} km")
    print("=" * 65)

    # Biểu đồ 1: Phân phối giá bán
    fig1 = px.histogram(
        sub, x='price_million', nbins=30, marginal='box', color_discrete_sequence=['#1f77b4'],
        title=f"Phân Phối Giá Xe (Trung vị: {med_p:.1f} Triệu VNĐ)",
        labels={'price_million': 'Giá bán (Triệu VNĐ)'}
    )
    fig1.update_layout(height=340, margin=dict(l=40, r=20, t=40, b=30), yaxis_title="Số lượng xe")

    # Biểu đồ 2: Tương quan ODO và Giá bán theo năm
    fig2 = px.scatter(
        sub, x='mileage_v2', y='price_million', color='mfdate', color_continuous_scale='Turbo',
        title="Tương Quan ODO - Giá Bán Theo Năm Sản Xuất",
        labels={'mileage_v2': 'ODO (km)', 'price_million': 'Giá bán (Triệu VNĐ)', 'mfdate': 'Năm SX'},
        hover_data=['carmodel_name', 'region_name']
    )
    fig2.update_layout(height=360, margin=dict(l=40, r=20, t=40, b=30))

    fig1.show()
    fig2.show()

# 3. Kết nối tương tác và hiển thị giao diện
dashboard_out = widgets.interactive_output(
    render_dashboard,
    {'brand': w_brand, 'model': w_model, 'province': w_province, 'years': w_year}
)

controls = widgets.VBox([
    widgets.HBox([w_brand, w_model]),
    widgets.HBox([w_province, w_year, btn_reset])
])

display(controls, dashboard_out)


Output()